In [13]:
import pybedtools, os, re, pickle
import pandas as pd
import numpy as np

from matplotlib.gridspec import GridSpec
import matplotlib.gridspec as gridspec

from matplotlib import pyplot as plt
import seaborn as sns
import matplotlib

## 1. Consolidate Synthetic PWM results

In [14]:
############### PWM ###############
def pwm_main():
    pwm_info = pd.read_table(f"00_constructing_PWMs/simulated_ICcontent.csv.gz", sep=",")
    pwm_info["motif"] = [f"{i}_{i}" for i in pwm_info["TF"]]
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        #print(cl)
        base = "../05_footprinting/01_original/simulated/pwm"
        df = pd.read_table(f"{base}/{cl}_filt_500bp.exclusion_PWMmatch.bed")
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.reset_index(inplace=True)
        df.columns = ["motif", cl]
        pwm_info = pwm_info.merge(df, on="motif", how='left')
        
    pwm_info= pwm_info.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_info.to_csv('01_inputfiles/01_simulated_results_PWM.csv', index=False)

#pwm_main()
pwm_info = pd.read_table(f"01_inputfiles/01_simulated_results_PWM.csv.gz", sep=",")
display(pwm_info)

,TF,IC,len,GC_content,Repeat_content,chr10_prevalence,motif,HEPG2,MCF7,K562,GM12878,SKNSH
0,FAKE_ALL_1,9.669563,6,0.676648,0.516852,324485.0,FAKE_ALL_1_FAKE_ALL_1,727242.0,611471.0,797947.0,716541.0,699383.0
1,FAKE_ALL_10,7.677256,7,0.320848,0.401730,892646.0,FAKE_ALL_10_FAKE_ALL_10,1101270.0,989841.0,1242648.0,1363022.0,1155354.0
2,FAKE_ALL_100,23.403398,22,0.447950,0.310389,20145.0,FAKE_ALL_100_FAKE_ALL_100,30228.0,23417.0,41194.0,35884.0,30796.0
3,FAKE_ALL_101,24.889408,22,0.601676,0.337863,1501.0,FAKE_ALL_101_FAKE_ALL_101,4700.0,3893.0,4978.0,4407.0,4257.0
4,FAKE_ALL_102,22.341883,22,0.415137,0.332184,7194.0,FAKE_ALL_102_FAKE_ALL_102,9767.0,8814.0,10669.0,11497.0,9808.0
...,...,...,...,...,...,...,...,...,...,...,...,...
257,FAKE_USF_4,10.663675,8,0.759868,0.473490,34878.0,FAKE_USF_4_FAKE_USF_4,148890.0,120135.0,165603.0,141249.0,142218.0
258,FAKE_USF_5,6.557005,8,0.666551,0.446097,674997.0,FAKE_USF_5_FAKE_USF_5,1566473.0,1230253.0,1877388.0,1622930.0,1445485.0
259,FAKE_USF_6,9.013403,8,0.771092,0.535571,287642.0,FAKE_USF_6_FAKE_USF_6,991559.0,797990.0,1063234.0,929753.0,909977.0
260,FAKE_USF_7,8.695068,10,0.704818,0.422346,188095.0,FAKE_USF_7_FAKE_USF_7,644144.0,529995.0,706033.0,616344.0,604419.0


In [15]:
############### HINT ###############

def hint_main():
    pwm_info = pd.read_table(f"00_constructing_PWMs/simulated_ICcontent.csv.gz", sep=",")
    pwm_info["motif"] = [f"{i}_{i}" for i in pwm_info["TF"]]
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        print(cl)
        base = "../05_footprinting/01_original/simulated/hint"
        df = pd.read_table(f"{base}/{cl}-cellFilt_mpbs.bed",
                      header=None, sep="\t", usecols = [0,1,2,3, 4, 5], 
                            names=['chrom','start','end',"motif", "pwm_score","strand"])
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.reset_index(inplace=True)
        df.columns = ["TF", cl]
        pwm_info = pwm_info.merge(df, on="TF", how='left')
        
    pwm_info= pwm_info.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_info.to_csv('01_inputfiles/01_simulated_results_HINT.csv', index=False)

#hint_main()
pwm_info = pd.read_table(f"01_inputfiles/01_simulated_results_HINT.csv.gz", sep=",")
display(pwm_info)

,TF,IC,len,GC_content,Repeat_content,chr10_prevalence,motif,HEPG2,MCF7,K562,GM12878,SKNSH
0,FAKE_ALL_1,9.669563,6,0.676648,0.516852,324485.0,FAKE_ALL_1_FAKE_ALL_1,8958.0,7121.0,8177.0,7395.0,7938.0
1,FAKE_ALL_10,7.677256,7,0.320848,0.401730,892646.0,FAKE_ALL_10_FAKE_ALL_10,3915.0,3921.0,4218.0,6283.0,5080.0
2,FAKE_ALL_100,23.403398,22,0.447950,0.310389,20145.0,FAKE_ALL_100_FAKE_ALL_100,7370.0,5343.0,9830.0,9788.0,7772.0
3,FAKE_ALL_101,24.889408,22,0.601676,0.337863,1501.0,FAKE_ALL_101_FAKE_ALL_101,1345.0,1008.0,1344.0,1329.0,1178.0
4,FAKE_ALL_102,22.341883,22,0.415137,0.332184,7194.0,FAKE_ALL_102_FAKE_ALL_102,1860.0,1734.0,2058.0,2388.0,1943.0
...,...,...,...,...,...,...,...,...,...,...,...,...
257,FAKE_USF_4,10.663675,8,0.759868,0.473490,34878.0,FAKE_USF_4_FAKE_USF_4,1358.0,1062.0,1318.0,1237.0,1255.0
258,FAKE_USF_5,6.557005,8,0.666551,0.446097,674997.0,FAKE_USF_5_FAKE_USF_5,12011.0,10590.0,11650.0,12672.0,10799.0
259,FAKE_USF_6,9.013403,8,0.771092,0.535571,287642.0,FAKE_USF_6_FAKE_USF_6,12526.0,9408.0,11980.0,12023.0,10464.0
260,FAKE_USF_7,8.695068,10,0.704818,0.422346,188095.0,FAKE_USF_7_FAKE_USF_7,2777.0,2107.0,2626.0,2501.0,2567.0


In [16]:
############### TOBIAS ###############
def tobias_main():
    pwm_info = pd.read_table(f"00_constructing_PWMs/simulated_ICcontent.csv.gz", sep=",")
    pwm_info["motif"] = [f"{i}_{i}" for i in pwm_info["TF"]]
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        print(cl)
        base = "../05_footprinting/01_original/simulated/tobias"
        df = pd.read_table(f"{base}/{cl}-cellFilt_bindetect_TF_overviews.txt",
                                header=None, sep="\t", usecols = [0,1,2,3, 4, 5, 12, 13], 
                                names=['chrom','start','end',"motif", "pwm_score","strand", "score", "bound_bool"])
        df = df.loc[df['bound_bool'] == 1]
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.reset_index(inplace=True)
        df.columns = ["motif", cl]
        pwm_info = pwm_info.merge(df, on="motif", how='left')
        
    pwm_info= pwm_info.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_info.to_csv('01_inputfiles/01_simulated_results_TOBIAS.csv', index=False)

#tobias_main()
pwm_info = pd.read_table(f"01_inputfiles/01_simulated_results_TOBIAS.csv.gz", sep=",")
display(pwm_info)

,TF,IC,len,GC_content,Repeat_content,chr10_prevalence,motif,HEPG2,MCF7,K562,GM12878,SKNSH
0,FAKE_ALL_1,9.669563,6,0.676648,0.516852,324485.0,FAKE_ALL_1_FAKE_ALL_1,13054.0,12356.0,12924.0,32253.0,13528.0
1,FAKE_ALL_10,7.677256,7,0.320848,0.401730,892646.0,FAKE_ALL_10_FAKE_ALL_10,1680.0,2034.0,1934.0,25939.0,2836.0
2,FAKE_ALL_100,23.403398,22,0.447950,0.310389,20145.0,FAKE_ALL_100_FAKE_ALL_100,4188.0,4304.0,5329.0,63947.0,5628.0
3,FAKE_ALL_101,24.889408,22,0.601676,0.337863,1501.0,FAKE_ALL_101_FAKE_ALL_101,4051.0,4330.0,4119.0,18248.0,4971.0
4,FAKE_ALL_102,22.341883,22,0.415137,0.332184,7194.0,FAKE_ALL_102_FAKE_ALL_102,2752.0,2775.0,2890.0,19091.0,3006.0
...,...,...,...,...,...,...,...,...,...,...,...,...
257,FAKE_USF_4,10.663675,8,0.759868,0.473490,34878.0,FAKE_USF_4_FAKE_USF_4,3623.0,3793.0,3714.0,6830.0,4674.0
258,FAKE_USF_5,6.557005,8,0.666551,0.446097,674997.0,FAKE_USF_5_FAKE_USF_5,6886.0,9293.0,7230.0,45127.0,9335.0
259,FAKE_USF_6,9.013403,8,0.771092,0.535571,287642.0,FAKE_USF_6_FAKE_USF_6,11882.0,13798.0,11793.0,64285.0,16044.0
260,FAKE_USF_7,8.695068,10,0.704818,0.422346,188095.0,FAKE_USF_7_FAKE_USF_7,7860.0,9612.0,8392.0,21936.0,13347.0


In [17]:
############### PRINT ###############
def print_main():
    pwm_info = pd.read_table(f"00_constructing_PWMs/simulated_ICcontent.csv", sep=",")
    pwm_info["motif"] = pwm_info["TF"]
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        print(cl)
        base = "../05_footprinting/01_original/simulated/print"
        df = pd.read_table(f"{base}/{cl}-cellFilt_granges.bed",
                           header=0, sep="\t", 
                           usecols = [0,1,2,6,7], 
                           names=['chrom','start','end',"motif", "print_score"])
        df = df[df["print_score"] > 0.3]
        
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.reset_index(inplace=True)
        df.columns = ["motif", cl]
        pwm_info = pwm_info.merge(df, on="motif", how='left')
        
    pwm_info= pwm_info.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_info["motif"] = [f"{i}_{i}" for i in pwm_info["TF"]] #just to make it match others. 
    pwm_info.to_csv('01_inputfiles/01_simulated_results_PRINT.csv', index=False)

#print_main()
pwm_info = pd.read_table(f"01_inputfiles/01_simulated_results_PRINT.csv.gz", sep=",")
display(pwm_info)

,TF,IC,len,GC_content,Repeat_content,chr10_prevalence,motif,HEPG2,MCF7,K562,GM12878,SKNSH
0,FAKE_ALL_1,9.669563,6,0.676648,0.516852,324485.0,FAKE_ALL_1_FAKE_ALL_1,13124.0,12832.0,11428.0,8260.0,10013.0
1,FAKE_ALL_10,7.677256,7,0.320848,0.401730,892646.0,FAKE_ALL_10_FAKE_ALL_10,4344.0,5685.0,3897.0,5991.0,4916.0
2,FAKE_ALL_100,23.403398,22,0.447950,0.310389,20145.0,FAKE_ALL_100_FAKE_ALL_100,8146.0,7676.0,9927.0,6816.0,6594.0
3,FAKE_ALL_101,24.889408,22,0.601676,0.337863,1501.0,FAKE_ALL_101_FAKE_ALL_101,2211.0,2343.0,2075.0,2011.0,1873.0
4,FAKE_ALL_102,22.341883,22,0.415137,0.332184,7194.0,FAKE_ALL_102_FAKE_ALL_102,3377.0,3491.0,2961.0,3056.0,2535.0
...,...,...,...,...,...,...,...,...,...,...,...,...
257,FAKE_USF_4,10.663675,8,0.759868,0.473490,34878.0,FAKE_USF_4_FAKE_USF_4,2014.0,1799.0,2018.0,1849.0,1855.0
258,FAKE_USF_5,6.557005,8,0.666551,0.446097,674997.0,FAKE_USF_5_FAKE_USF_5,9382.0,10709.0,8338.0,7286.0,7197.0
259,FAKE_USF_6,9.013403,8,0.771092,0.535571,287642.0,FAKE_USF_6_FAKE_USF_6,8750.0,13716.0,7726.0,9676.0,6201.0
260,FAKE_USF_7,8.695068,10,0.704818,0.422346,188095.0,FAKE_USF_7_FAKE_USF_7,3245.0,3033.0,3254.0,2736.0,3115.0


## 2. Consolidate JASPAR PWM results

In [18]:
##LOAD IN JASPAR INFO
pwm_info = pd.read_table("../05_footprinting/program_input_files/jaspar2022_ICcontent.csv.gz", sep=",")
pwm_info.TF = pwm_info.TF.astype(str).str.upper()
pwm_info["motif"] = pwm_info.TF
pwm_info["TF"] = [i.rsplit("_",1)[0] for i in pwm_info.TF]

##LOAD IN USF data
usf_df = pd.read_table("../05_footprinting/program_input_files/USF_list.csv.gz", sep=",", header=None) 
pwm_info["USF"] = 0
pwm_info.loc[pwm_info["TF"].isin(usf_df[0].values), "USF"] = 1

#LOAD IN RNAseq INFO
rnaseq = "../05_footprinting/program_input_files/ENCODE_polyAplus.csv.gz"
rnaseq_df = pd.read_csv(rnaseq, sep=",")
rnaseq_df.columns = [f"{i}_expression" for i in rnaseq_df.columns]

pwm_info = pwm_info.merge(rnaseq_df, left_on="TF", right_on="GeneSymbol_expression", how='left') #get all the easy ones. 
not_found = pwm_info[pwm_info["HEPG2_expression"].isnull()]["TF"]
for tf in not_found:
    tf_split = tf.rsplit("::",1)
    vals = rnaseq_df[rnaseq_df["GeneSymbol_expression"].isin(tf_split)].sum()
    pwm_info.loc[pwm_info["TF"] == tf, list(vals.index)] = vals.values

pwm_info.drop("GeneSymbol_expression", axis=1, inplace=True)
display(pwm_info)

,TF,IC,len,obs,GC_content,Repeat_content,chr10_prevalence,motif,USF,HEPG2_expression,K562_expression,GM12878_expression,MCF7_expression,SKNSH_expression
0,AHR::ARNT,8.052948,6,24,0.708333,0.491935,397442,AHR::ARNT_MA0006.1,0,28.753636,23.132500,20.545833,42.1175,17.972
1,ALX1,12.902574,17,100,0.333954,0.336005,250084,ALX1_MA0854.1,0,0.004545,0.100625,0.009167,0.0000,0.476
2,ALX3,7.447932,10,7877,0.275671,0.368898,1130741,ALX3_MA0634.1,0,0.000000,0.010625,0.002500,0.0000,0.106
3,ALX4,12.807436,17,101,0.349303,0.326209,229630,ALX4_MA0853.1,0,0.004545,0.292500,0.000833,0.0475,0.018
4,AR,18.630341,17,3102,0.519749,0.275296,18170,AR_MA0007.3,0,0.000000,0.005000,0.020833,5.3775,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,ZNF85,15.359574,16,584,0.419310,0.338706,192254,ZNF85_MA1720.1,0,2.010909,0.800625,14.778333,10.8650,2.976
837,ZNF93,16.175846,16,2388,0.745678,0.468049,72871,ZNF93_MA1721.1,0,0.508182,8.898750,2.782500,6.6725,4.498
838,ZSCAN29,14.774329,12,999,0.568034,0.325625,32204,ZSCAN29_MA1602.1,0,12.391818,10.113125,7.948333,9.1375,10.442
839,ZSCAN31,24.794394,19,8838,0.571115,0.365572,2668,ZSCAN31_MA1722.1,0,5.612727,5.511250,2.020833,3.4275,2.644


In [19]:
############### PWM ###############

def pwm_main():
    pwm_copy = pwm_info.copy()
    pwm_copy["merge_on"] = [f"{i}_{i}" for i in pwm_copy.motif]
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        print(cl)
        base = "../05_footprinting/01_original/pwm"
        df = pd.read_table(f"{base}/{cl}_filt_500bp.exclusion_PWMmatch.bed")
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.reset_index(inplace=True)
        df.columns = ["merge_on", cl]
        df.merge_on = df.merge_on.astype(str).str.upper()
        pwm_copy = pwm_copy.merge(df, on="merge_on", how='left')
        
    pwm_copy= pwm_copy.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_copy.to_csv('01_inputfiles/01_jaspar_results_PWM.csv', index=False)

#pwm_main()
df = pd.read_table(f"01_inputfiles/01_jaspar_results_PWM.csv.gz", sep=",")
display(df)

,TF,IC,len,obs,GC_content,Repeat_content,chr10_prevalence,motif,USF,HEPG2_expression,GM12878_expression,K562_expression,merge_on,HEPG2,MCF7,K562,GM12878,SKNSH
0,AHR::ARNT,8.052948,6,24,0.708333,0.491935,397442,AHR::ARNT_MA0006.1,0,23.3845,18.0835,17.870938,AHR::ARNT_MA0006.1_AHR::ARNT_MA0006.1,749324,623372,883801,831619,736138
1,ALX1,12.902574,17,100,0.333954,0.336005,250084,ALX1_MA0854.1,0,0.9885,0.0095,0.065312,ALX1_MA0854.1_ALX1_MA0854.1,204566,204460,228854,288315,251542
2,ALX3,7.447932,10,7877,0.275671,0.368898,1130741,ALX3_MA0634.1,0,0.0000,0.0015,0.005313,ALX3_MA0634.1_ALX3_MA0634.1,972916,944836,1125701,1343293,1182309
3,ALX4,12.807436,17,101,0.349303,0.326209,229630,ALX4_MA0853.1,0,0.0025,0.0005,0.249375,ALX4_MA0853.1_ALX4_MA0853.1,194145,193374,213959,272097,238411
4,AR,18.630341,17,3102,0.519749,0.275296,18170,AR_MA0007.3,0,0.0005,0.0150,0.005000,AR_MA0007.3_AR_MA0007.3,37080,32134,39543,37981,35732
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,ZNF85,15.359574,16,584,0.419310,0.338706,192254,ZNF85_MA1720.1,0,1.4695,10.7035,0.653125,ZNF85_MA1720.1_ZNF85_MA1720.1,288661,228653,357199,326578,279373
837,ZNF93,16.175846,16,2388,0.745678,0.468049,72871,ZNF93_MA1721.1,0,0.3450,3.0070,7.427187,ZNF93_MA1721.1_ZNF93_MA1721.1,355564,295547,370063,329214,335002
838,ZSCAN29,14.774329,12,999,0.568034,0.325625,32204,ZSCAN29_MA1602.1,0,10.1180,7.5915,8.703125,ZSCAN29_MA1602.1_ZSCAN29_MA1602.1,63564,54712,72082,65833,60635
839,ZSCAN31,24.794394,19,8838,0.571115,0.365572,2668,ZSCAN31_MA1722.1,0,4.0575,1.5900,4.339062,ZSCAN31_MA1722.1_ZSCAN31_MA1722.1,5117,4565,5574,5468,5085


In [20]:
############### HINT ###############

def hint_main():
    pwm_copy = pwm_info.copy()
    pwm_copy["merge_on"] = pwm_copy.motif
    
    def quick_fun(tmp):
        a = tmp.rsplit('.', 1)
        b = f"{a[1]}_{a[0]}"
        return(b.upper())
    
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        print(cl)
        base = "../05_footprinting/01_original/hint"
        df = pd.read_table(f"{base}/{cl}-cellFilt_mpbs.bed",
                      header=None, sep="\t", usecols = [0,1,2,3, 4, 5], 
                            names=['chrom','start','end',"motif", "pwm_score","strand"])
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.index = [quick_fun(i) for i in df.index]
        df.reset_index(inplace=True)
        df.columns = ["merge_on", cl]
        pwm_copy = pwm_copy.merge(df, on="merge_on", how='left')
        
    pwm_copy= pwm_copy.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_copy.to_csv('01_inputfiles/01_jaspar_results_HINT.csv', index=False)

#hint_main()
df = pd.read_table(f"01_inputfiles/01_jaspar_results_HINT.csv.gz", sep=",")
display(df)

,TF,IC,len,obs,GC_content,Repeat_content,chr10_prevalence,motif,USF,HEPG2_expression,GM12878_expression,K562_expression,merge_on,HEPG2,MCF7,K562,GM12878,SKNSH
0,AHR::ARNT,8.052948,6,24,0.708333,0.491935,397442,AHR::ARNT_MA0006.1,0,23.3845,18.0835,17.870938,AHR::ARNT_MA0006.1,8345,6540,9561,9386,7692
1,ALX1,12.902574,17,100,0.333954,0.336005,250084,ALX1_MA0854.1,0,0.9885,0.0095,0.065312,ALX1_MA0854.1,1850,1900,1927,3648,2697
2,ALX3,7.447932,10,7877,0.275671,0.368898,1130741,ALX3_MA0634.1,0,0.0000,0.0015,0.005313,ALX3_MA0634.1,3044,3753,3704,5946,4959
3,ALX4,12.807436,17,101,0.349303,0.326209,229630,ALX4_MA0853.1,0,0.0025,0.0005,0.249375,ALX4_MA0853.1,1588,1616,1608,3100,2343
4,AR,18.630341,17,3102,0.519749,0.275296,18170,AR_MA0007.3,0,0.0005,0.0150,0.005000,AR_MA0007.3,2844,2558,2893,3359,2653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,ZNF85,15.359574,16,584,0.419310,0.338706,192254,ZNF85_MA1720.1,0,1.4695,10.7035,0.653125,ZNF85_MA1720.1,8445,6207,8819,9450,7125
837,ZNF93,16.175846,16,2388,0.745678,0.468049,72871,ZNF93_MA1721.1,0,0.3450,3.0070,7.427187,ZNF93_MA1721.1,13031,9652,11698,11882,11659
838,ZSCAN29,14.774329,12,999,0.568034,0.325625,32204,ZSCAN29_MA1602.1,0,10.1180,7.5915,8.703125,ZSCAN29_MA1602.1,2574,2162,2636,2521,2154
839,ZSCAN31,24.794394,19,8838,0.571115,0.365572,2668,ZSCAN31_MA1722.1,0,4.0575,1.5900,4.339062,ZSCAN31_MA1722.1,2712,2411,2659,2984,2553


In [21]:
############### TOBIAS ###############
def tobias_main():
    pwm_copy = pwm_info.copy()
    pwm_copy["merge_on"] = [f"{i}_{i}" for i in pwm_copy.motif]
    pwm_copy["merge_on"] = [i.replace(":","") for i in pwm_copy.merge_on]
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        print(cl)
        base = "../05_footprinting/01_original/tobias"
        df = pd.read_table(f"{base}/{cl}-cellFilt_bindetect_TF_overviews.txt",
                                header=None, sep="\t", usecols = [0,1,2,3, 4, 5, 12, 13], 
                                names=['chrom','start','end',"motif", "pwm_score","strand", "score", "bound_bool"])
        df = df.loc[df['bound_bool'] == 1]
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.reset_index(inplace=True)
        df.columns = ["merge_on", cl]
        df.merge_on = df.merge_on.astype(str).str.upper()
        pwm_copy = pwm_copy.merge(df, on="merge_on", how='left')
        
    pwm_copy = pwm_copy.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_copy.to_csv('01_inputfiles/01_jaspar_results_TOBIAS.csv', index=False)

#tobias_main()
df = pd.read_table(f"01_inputfiles/01_jaspar_results_TOBIAS.csv.gz", sep=",")
display(df)

,TF,IC,len,obs,GC_content,Repeat_content,chr10_prevalence,motif,USF,HEPG2_expression,GM12878_expression,K562_expression,merge_on,HEPG2,MCF7,K562,GM12878,SKNSH
0,AHR::ARNT,8.052948,6,24,0.708333,0.491935,397442,AHR::ARNT_MA0006.1,0,23.3845,18.0835,17.870938,AHRARNT_MA0006.1_AHRARNT_MA0006.1,6791,7976,7764,32358,7773
1,ALX1,12.902574,17,100,0.333954,0.336005,250084,ALX1_MA0854.1,0,0.9885,0.0095,0.065312,ALX1_MA0854.1_ALX1_MA0854.1,1142,1230,1307,14158,1602
2,ALX3,7.447932,10,7877,0.275671,0.368898,1130741,ALX3_MA0634.1,0,0.0000,0.0015,0.005313,ALX3_MA0634.1_ALX3_MA0634.1,1220,1651,1698,17620,2365
3,ALX4,12.807436,17,101,0.349303,0.326209,229630,ALX4_MA0853.1,0,0.0025,0.0005,0.249375,ALX4_MA0853.1_ALX4_MA0853.1,1003,1133,1193,12149,1502
4,AR,18.630341,17,3102,0.519749,0.275296,18170,AR_MA0007.3,0,0.0005,0.0150,0.005000,AR_MA0007.3_AR_MA0007.3,2636,2628,2674,17902,2862
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,ZNF85,15.359574,16,584,0.419310,0.338706,192254,ZNF85_MA1720.1,0,1.4695,10.7035,0.653125,ZNF85_MA1720.1_ZNF85_MA1720.1,7033,7896,7622,74324,6221
837,ZNF93,16.175846,16,2388,0.745678,0.468049,72871,ZNF93_MA1721.1,0,0.3450,3.0070,7.427187,ZNF93_MA1721.1_ZNF93_MA1721.1,26572,27774,27575,80538,35067
838,ZSCAN29,14.774329,12,999,0.568034,0.325625,32204,ZSCAN29_MA1602.1,0,10.1180,7.5915,8.703125,ZSCAN29_MA1602.1_ZSCAN29_MA1602.1,2560,2984,2787,14044,2648
839,ZSCAN31,24.794394,19,8838,0.571115,0.365572,2668,ZSCAN31_MA1722.1,0,4.0575,1.5900,4.339062,ZSCAN31_MA1722.1_ZSCAN31_MA1722.1,4495,5041,4751,31669,5596


In [22]:
############### PRINT ###############

def print_main():
    pwm_copy = pwm_info.copy()
    pwm_copy["merge_on"] = pwm_copy.motif
    
    for cl in ["HEPG2", "MCF7", "K562", "GM12878", "SKNSH"]:
        print(cl)
        base = "../05_footprinting/01_original/print"
        df = pd.read_table(f"{base}/{cl}_granges.bed",
                           header=0, sep="\t", 
                           usecols = [0,1,2,6,7], 
                           names=['chrom','start','end',"motif", "print_score"])
        df = df[df["print_score"] > 0.3]
        
        df = df.groupby(['motif'])["motif"].agg(['count'])
        df.reset_index(inplace=True)
        df.columns = ["merge_on", cl]
        df.merge_on = df.merge_on.astype(str).str.upper()
        pwm_copy = pwm_copy.merge(df, on="merge_on", how='left')
        
    pwm_copy = pwm_copy.dropna(subset=["HEPG2", "MCF7", "K562","GM12878","SKNSH"])
    pwm_copy.to_csv('01_inputfiles/01_jaspar_results_PRINT.csv', index=False)

#print_main()
df = pd.read_table(f"01_inputfiles/01_jaspar_results_PRINT.csv.gz", sep=",")
display(df)

,TF,IC,len,obs,GC_content,Repeat_content,chr10_prevalence,motif,USF,HEPG2_expression,GM12878_expression,K562_expression,merge_on,HEPG2,MCF7,K562,GM12878,SKNSH
0,AHR::ARNT,8.052948,6,24,0.708333,0.491935,397442,AHR::ARNT_MA0006.1,0,23.3845,18.0835,17.870938,AHR::ARNT_MA0006.1,9462,9825,9738,8485,7471
1,ALX1,12.902574,17,100,0.333954,0.336005,250084,ALX1_MA0854.1,0,0.9885,0.0095,0.065312,ALX1_MA0854.1,1928,2396,1766,2842,2133
2,ALX3,7.447932,10,7877,0.275671,0.368898,1130741,ALX3_MA0634.1,0,0.0000,0.0015,0.005313,ALX3_MA0634.1,1898,3195,1930,2857,2932
3,ALX4,12.807436,17,101,0.349303,0.326209,229630,ALX4_MA0853.1,0,0.0025,0.0005,0.249375,ALX4_MA0853.1,1548,2100,1471,2354,1906
4,AR,18.630341,17,3102,0.519749,0.275296,18170,AR_MA0007.3,0,0.0005,0.0150,0.005000,AR_MA0007.3,4384,4803,3740,3764,3248
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
836,ZNF85,15.359574,16,584,0.419310,0.338706,192254,ZNF85_MA1720.1,0,1.4695,10.7035,0.653125,ZNF85_MA1720.1,10060,9892,9459,5756,5631
837,ZNF93,16.175846,16,2388,0.745678,0.468049,72871,ZNF93_MA1721.1,0,0.3450,3.0070,7.427187,ZNF93_MA1721.1,15358,16082,14746,15082,14844
838,ZSCAN29,14.774329,12,999,0.568034,0.325625,32204,ZSCAN29_MA1602.1,0,10.1180,7.5915,8.703125,ZSCAN29_MA1602.1,1917,2064,1804,1731,1522
839,ZSCAN31,24.794394,19,8838,0.571115,0.365572,2668,ZSCAN31_MA1722.1,0,4.0575,1.5900,4.339062,ZSCAN31_MA1722.1,3778,4524,3403,3495,3216
